# 长期记忆-agent

## 1、在工具中访问长期记忆


In [1]:

from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 1、提供大模型
load_dotenv(override=True)

model = init_chat_model(
    model="qwen3.7-plus",
    model_provider="openai",
    profile={"max_input_tokens": 128_000},
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    # temperature=1.5,
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    extra_body={"enable_thinking": False},
    # max_tokens=10,
)

### 1.1  基于InMemoryStore

In [2]:
from langchain_core.tools import tool
from typing import NotRequired
from langgraph.prebuilt import ToolRuntime
from langchain_core.messages import HumanMessage
from langgraph.store.memory import InMemoryStore
from langchain.agents import create_agent, AgentState


store = InMemoryStore()

# 自定义一个继承于AgentState的类
class CustomState(AgentState):
    user_id : NotRequired[str]

# 保存用户信息到长期记忆中
@tool(parse_docstring=True)
def save_user_info(name : str, runtime : ToolRuntime) -> str:
    """
    将客户信息保存在长期记忆中

    Args:
        name : 用户名
        runtime : 工具运行时

    Returns:
        str : 保存状态
    """
    namespace = ("user",)
    key = runtime.state["user_id"]
    value = {"name" : name}

    runtime.store.put(namespace, key, value)
    return "saved"

@tool(parse_docstring=True)
def get_user_info(runtime : ToolRuntime) -> str:
    """
    从长期记忆中读取客户的信息

    Args:
         runtime : 工具的运行时

    Returns:
        str : 用户信息
    """
    namespace = ("user",)
    key = runtime.state["user_id"]

    item = runtime.store.get(namespace, key)
    return str(item.value) if item else "unknow"

agent = create_agent(
    model = model,
    tools = [save_user_info, get_user_info],
    store = store,
    state_schema = CustomState,
    system_prompt = "用户提及个人信息的时候，可以使用工具保存用户信息，如果用户询问个人信息时，可以尝试使用工具读取用户信息。"
)

print("=" * 30, '-> 第一个会话（线程） <-', "=" * 30)
response1 = agent.invoke({
    "messages": [HumanMessage("你好，很高兴认识你，我是小花")],
    "user_id": "user-1"
})
for msg in response1["messages"]:
    msg.pretty_print()

print("=" * 30, '-> 第二个会话（线程） <-', "=" * 30)
response2 = agent.invoke({
    "messages": [HumanMessage("我是谁")],
    "user_id": "user-1"
})
for msg in response2["messages"]:
    msg.pretty_print()

============================== -> 第一个会话（线程） <- ==============================
================================ Human Message =================================

你好，很高兴认识你，我是小花
================================== Ai Message ==================================

你好，小花！很高兴认识你。
Tool Calls:
  save_user_info (call_c948accbd8964661938740d3)
 Call ID: call_c948accbd8964661938740d3
  Args:
    name: 小花
================================= Tool Message =================================
Name: save_user_info

saved
================================== Ai Message ==================================

我已经记住了你的名字。有什么我可以帮你的吗？
============================== -> 第二个会话（线程） <- ==============================
================================ Human Message =================================

我是谁
================================== Ai Message ==================================

我来帮您查看一下您的个人信息。
Tool Calls:
  get_user_info (call_dd856fee78de46c6ab036a83)
 Call ID: call_dd856fee78de46c6ab036a83
  Args:
=======================

### 1.2 基于PostgresStore

In [ ]:
from langchain_core.tools import tool
from typing import NotRequired
from langgraph.prebuilt import ToolRuntime
from langchain_core.messages import HumanMessage
from langgraph.store.memory import InMemoryStore
from langchain.agents import create_agent, AgentState
from langgraph.store.postgres import PostgresStore


# 自定义一个继承于AgentState的类
class CustomState(AgentState):
    user_id : NotRequired[str]

# 保存用户信息到长期记忆中
@tool(parse_docstring=True)
def save_user_info(name : str, runtime : ToolRuntime) -> str:
    """
    将客户信息保存在长期记忆中

    Args:
        name : 用户名
        runtime : 工具运行时

    Returns:
        str : 保存状态
    """
    namespace = ("user",)
    key = runtime.state["user_id"]
    value = {"name" : name}

    runtime.store.put(namespace, key, value)
    return "saved"

@tool(parse_docstring=True)
def get_user_info(runtime : ToolRuntime) -> str:
    """
    从长期记忆中读取客户的信息

    Args:
         runtime : 工具的运行时

    Returns:
        str : 用户信息
    """
    namespace = ("user",)
    key = runtime.state["user_id"]

    item = runtime.store.get(namespace, key)
    return str(item.value) if item else "unknow"


DB_URL = "you_DB_URL"
with PostgresStore.from_conn_string(DB_URL) as store:
    store.setup()

    agent = create_agent(
        model = model,
        tools = [save_user_info, get_user_info],
        store = store,
        state_schema = CustomState,
        system_prompt = "用户提及个人信息的时候，可以使用工具保存用户信息，如果用户询问个人信息时，可以尝试使用工具读取用户信息。"
    )

    print("=" * 30, '-> 第一个会话（线程） <-', "=" * 30)
    response1 = agent.invoke({
        "messages": [HumanMessage("你好，很高兴认识你，我是小花")],
        "user_id": "user-1"
    })
    for msg in response1["messages"]:
        msg.pretty_print()

    print("=" * 30, '-> 第二个会话（线程） <-', "=" * 30)
    response2 = agent.invoke({
        "messages": [HumanMessage("我是谁")],
        "user_id": "user-1"
    })
    for msg in response2["messages"]:
        msg.pretty_print()

## 2、在中间件中访问长期记忆

### 1、Node-style hooks中访问

以 before_model 为例，其钩子函数签名如下


In [ ]:
# def before_model(self, state: StateT, runtime: Runtime[ContextT]) -> dict[str, Any] | None:

Runtime 定义如下

In [ ]:
# @dataclass(**_DC_KWARGS)
# class Runtime(Generic[ContextT]):
#     context: ContextT = field(default=None)  # type: ignore[assignment]
#     """Static context for the graph run, like `user_id`, `db_conn`, etc.
#
#     Can also be thought of as 'run dependencies'."""
#
#     store: BaseStore | None = field(default=None)
#     """Store for the graph run, enabling persistence and memory."""
#
#     stream_writer: StreamWriter = field(default=_no_op_stream_writer)
#     """Function that writes to the custom stream."""
#
#     previous: Any = field(default=None)
#     """The previous return value for the given thread.
#
#     Only available with the functional API when a checkpointer is provided.
#     """

所以，我们可以通过 runtime.store 在中间件中访问长期记忆。

### 2、Wrap-style hooks中访问

1. wrap_model_call

    钩子函数签名如下

In [ ]:
# def wrap_model_call(
#     self,
#     request: ModelRequest[ContextT],
#     handler: Callable[[ModelRequest[ContextT]], ModelResponse[ResponseT]],
# ) -> ModelResponse[ResponseT] | AIMessage | ExtendedModelResponse[ResponseT]:

ModelRequest 定义如下

In [ ]:
# @dataclass(init=False)
# class ModelRequest(Generic[ContextT]):
#     """Model request information for the agent.
#     Type Parameters:
#         ContextT: The type of the runtime context. Defaults to `None` if not specified.
#     """
#     model: BaseChatModel
#     messages: list[AnyMessage]  # excluding system message
#     system_message: SystemMessage | None
#     tool_choice: Any | None
#     tools: list[BaseTool | dict[str, Any]]
#     response_format: ResponseFormat[Any] | None
#     state: AgentState[Any]
#     runtime: Runtime[ContextT]
#     model_settings: dict[str, Any] = field(default_factory=dict)

所以，可以通过 request.runtime.store 访问长期记忆。

2.wrap_tool_call

    钩子函数签名如下


In [ ]:
# def wrap_tool_call(
#     self,
#     request: ToolCallRequest,
#     handler: Callable[[ToolCallRequest], ToolMessage | Command[Any]],
# ) -> ToolMessage | Command[Any]:

ToolCallRequest 定义如下


In [ ]:
# @dataclass
# class ToolCallRequest:
#     """Tool execution request passed to tool call interceptors.
#     Attributes:
#         tool_call: Tool call dict with name, args, and id from model output.
#         tool: BaseTool instance to be invoked, or None if tool is not
#             registered with the `ToolNode`. When tool is `None`, interceptors can
#             handle the request without validation. If the interceptor calls `execute()`,
#             validation will occur and raise an error for unregistered tools.
#         state: Agent state (`dict`, `list`, or `BaseModel`).
#         runtime: ToolRuntime context (optional, `None` if outside graph).
#     """
#     tool_call: ToolCall
#     tool: BaseTool | None
#     state: Any
#     runtime: ToolRuntime

ToolRuntime 定义如下


In [ ]:
# @dataclass
# class ToolRuntime(_DirectlyInjectedToolArg, Generic[ContextT, StateT]):
#     state: StateT
#     context: ContextT
#     config: RunnableConfig
#     stream_writer: StreamWriter
#     tool_call_id: str | None
#     store: BaseStore | None